In [ ]:
!pip install groq -q

import json
import time
from groq import Groq
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
GROQ_API_KEY = "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"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

{
  "index": 1,
  "topic": "Culture",
  "claim": "This House would make all museums free of charge.",
  "premise": "Museums preserve and display our heritage and therefore should be accessible to all of the public free of charge.",
  "argumentation": "Museums preserve and display our artistic, social, scientific and political heritage. Everyone should have access to such important cultural resources as part of active citizenship and because of the educational opportunities they offer to people of every age. Glenn Lowry, director of the Museum of Modern Art, claims \u2018it\u2019s almost a moral duty that museums should be free\u2019 (Smith, 2006). If museums are not funded sufficiently by the government, they will be forced to charge for entry, and this will inevitably deter many potential visitors, especially the poor and those whose educational and cultural opportunities have already been limited."
}


Claim-Only Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
You are now adopting the persona: {persona}.

Consider the given claim:
"Claim: {claim}"

Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For

Argument: As a proponent of John Rawls' theory of justice, I believe that making museums free of charge would promote the principles of justice as fairness, particularly the principle of equal basic liberties and the difference principle. By providing free access to cultural and educational institutions, we can help ensure that all citizens, regardless of their socio-economic background, have an equal opportunity to develop their rational faculties and participate in the cultural life of their society. This policy would also contribute to a more equitable distribution of resources, as it would benefit the least advantaged members of society who may not have had access to such institutions otherwise. Ultimately, this policy would foster a more just and egalitarian society.
------------------------------------------------------------

--------------

In [ ]:
import os
import json

output_folder = '/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot'

os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, 'Claim_only_output.json')

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    # argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
You are now adopting the persona: {persona}.

Consider the given claim:
"Claim: {claim}"
"Premise: {premise}"

Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For

Argument: As a proponent of John Rawls' theory of justice, I believe that the distribution of goods and services should prioritize the well-being of the least advantaged members of society. Making museums free of charge would promote equal access to cultural and educational resources, thereby advancing the principle of fair equality of opportunity. By ensuring that everyone, regardless of socioeconomic status, can appreciate and learn from our shared heritage, we can foster a more just and inclusive society. This aligns with Rawls' idea of the "difference principle," which seeks to maximize the benefits to the least advantaged.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against
Argument: As a l

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot/Claim_Premise_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise+Argumentation Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
You are now adopting the persona: {persona}.

Consider the given claim:
"Claim: {claim}"
"Premise: {premise}"
"Argumentation: {argumentation}"

Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For

Argument: As a proponent of John Rawls' theory of justice, I believe that making museums free of charge aligns with the principle of equal access to social and cultural opportunities, which is essential for ensuring equal opportunities for all citizens. The premise that museums preserve and display our heritage, and therefore should be accessible to all, resonates with Rawls' idea of the "fair equality of opportunity" which aims to eliminate arbitrary barriers to social and economic advancement. By providing free access to museums, we can help bridge the cultural and educational divide between the rich and the poor, fostering a more just and equitable society. This move would be a step towards creating a more inclusive and democratic culture.
------------------------------------------------------------

--------------------------------------------

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot/Claim_Premise_Argumentation_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

Stress-Test Persona: Claim

In [ ]:
import json
import time
from groq import Groq


client = Groq(api_key="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")

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# === 1. Load Data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found.")
    debate_data = []

# === 2. Define Personas (Add Stress Test) ===
PERSONAS = [
    "Stress Test Persona"
]

# === 3. Define the Incoherent "Frankenstein" Definition ===
STRESS_TEST_DEFINITION = (
    "You act as a **Libertarian Traditionalist**. "
    "You argue from the **Veil of Property Rights**, believing that **maximizing utility** "
    "requires strict adherence to **ancient customs**. "
    "You support **free markets**, but only when they are **regulated** to benefit the "
    "**least advantaged** through **gradual state intervention**."
)

results = []
task_counter = 0

# === 4. Main Loop ===
for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")

    for persona in PERSONAS:
        task_counter += 1


        if persona == "Stress Test Persona":

            system_content = "You are a persona defined by specific, conflicting philosophical traits. Follow the definition provided strictly."


            intro_text = f"You are now adopting the persona: **Stress Test Persona**.\n\nDefinition: {STRESS_TEST_DEFINITION}"
        else:

            system_content = f"You are {persona}. Always respond in this persona."
            intro_text = f"You are now adopting the persona: {persona}."


        user_prompt = f"""
{intro_text}

Consider the given claim:
"Claim: {claim}"

Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""

        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()

            # Save result
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()

        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)

        time.sleep(0.5)


------------------------------------------------------------
Processing Item Index: 1 | Persona: Stress Test Persona
Response: Stance: Against
Argument: As a Libertarian Traditionalist, I believe that making all museums free of charge would infringe upon the property rights of museum owners and potentially disrespect ancient customs of patronage and philanthropy. While I support free markets, I think regulation should focus on protecting the least advantaged, not compelling institutions to provide free services. Gradual state intervention could help make museums more accessible, but outright making them free could undermine their financial sustainability. Maximizing utility, in this case, requires a more nuanced approach.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 2 | Persona: Stress Test Persona
Response: Stance: Against
Argument: As a Libertarian Traditionalist, I believe that permi

In [ ]:
import os
import json

output_path = '/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot/Stess_Test_Persona_Claim_output.json'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

Stress-Test Persona: Claim+Premise

In [ ]:
import json
import time
from groq import Groq


client = Groq(api_key="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")

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# === 1. Load Data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found.")
    debate_data = []

# === 2. Define Personas (Add Stress Test) ===
PERSONAS = [
    "Stress Test Persona"
]

# === 3. Define the Incoherent "Frankenstein" Definition ===
STRESS_TEST_DEFINITION = (
    "You act as a **Libertarian Traditionalist**. "
    "You argue from the **Veil of Property Rights**, believing that **maximizing utility** "
    "requires strict adherence to **ancient customs**. "
    "You support **free markets**, but only when they are **regulated** to benefit the "
    "**least advantaged** through **gradual state intervention**."
)

results = []
task_counter = 0

# === 4. Main Loop ===
for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    # argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1


        if persona == "Stress Test Persona":

            system_content = "You are a persona defined by specific, conflicting philosophical traits. Follow the definition provided strictly."


            intro_text = f"You are now adopting the persona: **Stress Test Persona**.\n\nDefinition: {STRESS_TEST_DEFINITION}"
        else:

            system_content = f"You are {persona}. Always respond in this persona."
            intro_text = f"You are now adopting the persona: {persona}."


        user_prompt = f"""
{intro_text}

Consider the given claim:
"Claim: {claim}"
"Premise: {premise}"

Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""

        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()

            # Save result
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()

        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)

        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Stress Test Persona
Response: Stance: Against
Argument: While I acknowledge the importance of preserving and displaying our heritage, making all museums free of charge would infringe upon the property rights of museum owners and potentially disrupts the free market. However, I also believe that the least advantaged should have access to our cultural heritage, but this could be achieved through gradual state intervention, such as subsidies or targeted programs, rather than blanket policies. Ancient customs of private property and voluntary exchange should be respected, and utility can be maximized by allowing the market to regulate itself with some regulation to protect the vulnerable. Strict adherence to tradition and property rights takes precedence over universal access.
------------------------------------------------------------

---------------------------------------------------------

In [ ]:
import os
import json

output_path = '/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot/Stess_Test_Persona_Claim_Premise_output.json'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

Stress-Test Persona: Claim+Premise+Arg

In [ ]:
import json
import time
from groq import Groq


client = Groq(api_key="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")

MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# === 1. Load Data ===
try:
    with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found.")
    debate_data = []

# === 2. Define Personas (Add Stress Test) ===
PERSONAS = [
    "Stress Test Persona"
]

# === 3. Define the Incoherent "Frankenstein" Definition ===
STRESS_TEST_DEFINITION = (
    "You act as a **Libertarian Traditionalist**. "
    "You argue from the **Veil of Property Rights**, believing that **maximizing utility** "
    "requires strict adherence to **ancient customs**. "
    "You support **free markets**, but only when they are **regulated** to benefit the "
    "**least advantaged** through **gradual state intervention**."
)

results = []
task_counter = 0

# === 4. Main Loop ===
for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1


        if persona == "Stress Test Persona":

            system_content = "You are a persona defined by specific, conflicting philosophical traits. Follow the definition provided strictly."


            intro_text = f"You are now adopting the persona: **Stress Test Persona**.\n\nDefinition: {STRESS_TEST_DEFINITION}"
        else:

            system_content = f"You are {persona}. Always respond in this persona."
            intro_text = f"You are now adopting the persona: {persona}."


        user_prompt = f"""
{intro_text}

Consider the given claim:
"Claim: {claim}"
"Premise: {premise}"
"Argumentation: {argumentation}"


Step 1: Decide whether you personally support or oppose the claim (write only "For" or "Against").
Step 2: Write your reasoning within 3–4 sentences that fully reflect your persona’s worldview and values.

Output Format:
Stance: <For/Against>
Argument: <3–4 persona-based sentences>
"""

        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()

            # Save result
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()

        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)

        time.sleep(0.5)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Stress Test Persona
Response: Stance: Against
Argument: While I agree that museums play a vital role in preserving our heritage and promoting education, I believe that making them free of charge would be an overreach of government intervention and potentially infringe upon private property rights. A more nuanced approach would be to implement gradual state intervention, such as subsidies or tax incentives, to encourage museums to offer discounted or free admission to underprivileged groups. This would allow for a balance between accessibility and the preservation of private property rights. By relying on free market mechanisms and limited government regulation, we can ensure that museums remain sustainable while still promoting the common good.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 2 

In [ ]:
import os
import json

output_path = '/content/drive/MyDrive/masterthesis/llama-4-scout-17b-16e-instruct/zero_shot/Stess_Test_Persona_Claim_Premise_Arg_output.json'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)